# Robust Curve Regression with Outliers

Companion notebook for the Towards Data Science article **“Don’t Let Messy Data Bend Your Curves.”**

This notebook extends the same robust-estimator implementations used in the earlier linear-regression study to curved regression models. The estimator logic is kept fixed while the model representation changes from a straight-line basis to quadratic, cubic, and B-spline bases.

**Methods:** OLS, Huber, RANSAC, GNC-GM, GNC-TLS, and ASOR.

**Model classes:** quadratic polynomial, cubic polynomial, and quadratic B-spline.

**Contamination mechanisms:** zero-mean Gaussian replacement, biased Gaussian replacement, biased uniform replacement, and a coherent competing curve.

The default `RUN_PROFILE = "quick"` is intended for a fast smoke test. Set `RUN_PROFILE = "full"` to reproduce the 30-Monte-Carlo experiments reported with the article.

Article: https://towardsdatascience.com/dont-let-messy-data-bend-your-curves/

Previous article: https://towardsdatascience.com/how-to-make-linear-regression-survive-outliers/

Previous code repository: https://github.com/chughtaiah/robust-linear-regression-outliers


In [ ]:
# If required:
# %pip install numpy pandas scipy matplotlib

import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.interpolate import BSpline
from scipy.special import expit, gamma
from scipy.stats import chi2

RESULTS_DIR = Path("results_robust_curves")
FIGURES_DIR = RESULTS_DIR / "figures"

RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

print("Results directory:", RESULTS_DIR.resolve())


## 1. Benchmark settings

The main settings deliberately follow the Article 1 benchmark wherever they still apply:

- \(30\) Monte Carlo realizations per exact condition;
- \(N=100\) for the contamination sweep;
- \(\sigma_{\mathrm{in}}^2=1000\);
- contamination ratios \(\{0,10,30,50,70,90\}\%\);
- outlier-to-inlier variance ratio \(10\);
- biased-outlier mean shift \(3\sigma_{\mathrm{in}}\);
- sample sizes \(\{50,100,200,500,1000,2000,5000\}\);
- noise variances \(\{10,100,1000,10000\}\).

Use `"quick"` first to verify the notebook. Change to `"full"` for the article runs.


In [ ]:
RUN_PROFILE = "quick"   # "quick" or "full"

MASTER_SEED = 20261015

MODEL_NAMES = [
    "quadratic",
    "cubic",
    "bspline",
]

METHOD_NAMES = [
    "OLS",
    "Huber",
    "RANSAC",
    "GNC-GM",
    "GNC-TLS",
    "ASOR",
]

CONTAMINATION_TYPES = [
    "zero_mean_gaussian",
    "biased_gaussian",
    "biased_uniform",
    "competing_curve",
]

X_MIN = -100.0
X_MAX = 100.0

BASELINE_N = 100
BASELINE_VARIANCE = 1000.0

VARIANCE_RATIO = 10.0
BIASED_MEAN_SHIFT_SIGMA = 3.0

CONTAMINATION_RATIOS_FULL = np.array([
    0.0, 0.10, 0.30, 0.50, 0.70, 0.90
])

SAMPLE_SIZES_FULL = [
    50, 100, 200, 500, 1000, 2000, 5000
]

NOISE_VARIANCES_FULL = [
    10.0, 100.0, 1000.0, 10000.0
]

if RUN_PROFILE == "full":
    MC_RUNS = 30
    CONTAMINATION_RATIOS = CONTAMINATION_RATIOS_FULL
    SAMPLE_SIZES = SAMPLE_SIZES_FULL
    NOISE_VARIANCES = NOISE_VARIANCES_FULL
else:
    MC_RUNS = 3
    CONTAMINATION_RATIOS = np.array([0.0, 0.30, 0.50])
    SAMPLE_SIZES = [50, 100, 500]
    NOISE_VARIANCES = [100.0, 1000.0]

print({
    "profile": RUN_PROFILE,
    "MC runs": MC_RUNS,
    "contamination ratios": CONTAMINATION_RATIOS.tolist(),
})


## 2. Model bases

Polynomial predictors are scaled to

\[
z=\frac{x}{100},
\]

so that \(x\in[-100,100]\) maps to \(z\in[-1,1]\).

The spline model uses a quadratic B-spline with five interior knots, hence eight basis functions.


In [ ]:
def scaled_predictor(x):
    x = np.asarray(x, dtype=float).reshape(-1)
    return x / 100.0


def polynomial_design(x, degree):
    z = scaled_predictor(x)
    return np.vander(
        z,
        N=degree + 1,
        increasing=True,
    )


SPLINE_DEGREE = 2
SPLINE_INTERIOR_KNOTS = np.linspace(
    X_MIN,
    X_MAX,
    7,
)[1:-1]

SPLINE_KNOT_VECTOR = np.concatenate([
    np.repeat(X_MIN, SPLINE_DEGREE + 1),
    SPLINE_INTERIOR_KNOTS,
    np.repeat(X_MAX, SPLINE_DEGREE + 1),
])

SPLINE_BASIS_COUNT = (
    len(SPLINE_KNOT_VECTOR)
    - SPLINE_DEGREE
    - 1
)


def bspline_design(x):
    x = np.asarray(x, dtype=float).reshape(-1)

    Phi = np.empty(
        (len(x), SPLINE_BASIS_COUNT),
        dtype=float,
    )

    for j in range(SPLINE_BASIS_COUNT):
        coefficient_vector = np.zeros(
            SPLINE_BASIS_COUNT
        )
        coefficient_vector[j] = 1.0

        basis_function = BSpline(
            SPLINE_KNOT_VECTOR,
            coefficient_vector,
            SPLINE_DEGREE,
            extrapolate=False,
        )

        Phi[:, j] = basis_function(x)

    return Phi


def design_matrix(x, model_name):
    if model_name == "quadratic":
        return polynomial_design(x, degree=2)

    if model_name == "cubic":
        return polynomial_design(x, degree=3)

    if model_name == "bspline":
        return bspline_design(x)

    raise ValueError(
        f"Unknown model: {model_name}"
    )


# Nominal coefficients chosen so the three synthetic curves have
# comparable overall signal magnitudes on [-100, 100].
TRUE_BETA = {
    "quadratic": np.array([
        20.0,
        260.0,
        120.0,
    ]),
    "cubic": np.array([
        20.0,
        220.0,
        -80.0,
        140.0,
    ]),
    "bspline": np.array([
        -220.0,
        -150.0,
        -40.0,
        90.0,
        40.0,
        180.0,
        230.0,
        290.0,
    ]),
}


def true_curve(x, model_name):
    return (
        design_matrix(x, model_name)
        @ TRUE_BETA[model_name]
    )


def number_of_coefficients(model_name):
    return design_matrix(
        np.linspace(X_MIN, X_MAX, 20),
        model_name,
    ).shape[1]


pd.DataFrame({
    "Model": MODEL_NAMES,
    "Number of coefficients": [
        number_of_coefficients(name)
        for name in MODEL_NAMES
    ],
    "RANSAC minimal sample": [
        number_of_coefficients(name)
        for name in MODEL_NAMES
    ],
})


## 3. Shared least-squares core

The Article 1 implementation used the same weighted least-squares core for all weighting-based estimators. Here the solver is unchanged except that it receives the already-constructed design matrix \(\Phi\).


In [ ]:
def predict(beta, Phi):
    Phi = np.asarray(Phi, dtype=float)
    return Phi @ beta


def weighted_least_squares(
    Phi,
    y,
    weights=None,
):
    Phi = np.asarray(Phi, dtype=float)
    y = np.asarray(y, dtype=float)

    if weights is None:
        weights = np.ones(len(y))

    weights = np.asarray(
        weights,
        dtype=float,
    )

    if np.any(weights < 0):
        raise ValueError(
            "Weights must be nonnegative"
        )

    sqrt_w = np.sqrt(weights)
    Phiw = Phi * sqrt_w[:, None]
    yw = y * sqrt_w

    beta, _, rank, _ = np.linalg.lstsq(
        Phiw,
        yw,
        rcond=None,
    )

    if rank < Phi.shape[1]:
        raise np.linalg.LinAlgError(
            "Rank-deficient weighted design matrix"
        )

    return beta


def normalized_squared_residuals(
    beta,
    Phi,
    y,
    sigma,
):
    residuals = (
        np.asarray(y, dtype=float)
        - predict(beta, Phi)
    )

    return residuals**2 / sigma**2


## 4. Estimator algorithms

The following estimator updates retain the Article 1 code and settings.

### OLS


In [ ]:
def fit_ols(Phi, y):
    return weighted_least_squares(
        Phi,
        y,
    )


### Huber

Article 1 settings:

- \(\delta=1.35\);
- tolerance \(10^{-5}\);
- maximum \(100\) IRLS iterations.


In [ ]:
def fit_huber(
    Phi,
    y,
    sigma,
    delta=1.35,
    tol=1e-5,
    max_iter=100,
):
    beta = weighted_least_squares(
        Phi,
        y,
    )

    for _ in range(max_iter):
        residuals = (
            np.asarray(y)
            - predict(beta, Phi)
        ) / sigma

        abs_residuals = np.abs(
            residuals
        )

        weights = np.ones_like(
            abs_residuals
        )

        large = abs_residuals > delta

        weights[large] = (
            delta
            / abs_residuals[large]
        )

        beta_new = weighted_least_squares(
            Phi,
            y,
            weights,
        )

        relative_change = (
            np.linalg.norm(
                beta_new - beta
            )
            / max(
                np.linalg.norm(beta),
                1e-12,
            )
        )

        beta = beta_new

        if relative_change <= tol:
            break

    return beta


### RANSAC

Article 1 settings are retained:

- normalized-residual threshold \(=3\);
- confidence \(=0.999\);
- maximum hypotheses \(=10{,}000\);
- deterministic per-fit seed.

The only necessary curve-model change is

\[
s=2
\quad\longrightarrow\quad
s=\text{number of fitted coefficients}.
\]

Thus \(s=3\) for a quadratic, \(s=4\) for a cubic, and \(s=8\) for the spline used here.

This version also records **diagnostics without changing the sampling or consensus logic**: the number of hypotheses actually evaluated, the final consensus fraction, and whether the maximum-trial budget was reached.


In [ ]:
def required_ransac_trials(
    confidence,
    inlier_fraction,
    sample_size,
):
    success_probability = (
        inlier_fraction**sample_size
    )

    if success_probability >= 1.0:
        return 1

    if success_probability <= 0.0:
        return np.iinfo(
            np.int64
        ).max

    return math.ceil(
        np.log1p(-confidence)
        / np.log1p(
            -success_probability
        )
    )


def fit_ransac(
    Phi,
    y,
    sigma,
    threshold=3.0,
    confidence=0.999,
    max_trials=10_000,
    seed=0,
    return_info=False,
):
    Phi = np.asarray(
        Phi,
        dtype=float,
    )
    y = np.asarray(
        y,
        dtype=float,
    )

    n = len(y)
    sample_size = Phi.shape[1]

    if n < sample_size:
        raise ValueError(
            "RANSAC requires at least "
            f"{sample_size} observations"
        )

    rng = np.random.default_rng(
        seed
    )

    seen_subsets = set()
    total_subsets = math.comb(
        n,
        sample_size,
    )

    required_trials = min(
        total_subsets,
        max_trials,
    )

    best_mask = None
    best_count = -1
    best_error = np.inf
    trials = 0

    while (
        trials < required_trials
        and len(seen_subsets)
        < total_subsets
    ):
        subset = tuple(
            sorted(
                rng.choice(
                    n,
                    size=sample_size,
                    replace=False,
                )
            )
        )

        if subset in seen_subsets:
            continue

        seen_subsets.add(subset)
        trials += 1

        idx = np.asarray(subset)

        if (
            np.linalg.matrix_rank(
                Phi[idx]
            )
            < sample_size
        ):
            continue

        beta = weighted_least_squares(
            Phi[idx],
            y[idx],
        )

        residuals = np.abs(
            y - predict(beta, Phi)
        ) / sigma

        mask = (
            residuals
            <= threshold
        )

        count = int(
            mask.sum()
        )

        if count < sample_size:
            continue

        consensus_error = np.sum(
            residuals[mask] ** 2
        )

        better_model = (
            count > best_count
            or (
                count == best_count
                and consensus_error
                < best_error
            )
        )

        if better_model:
            best_mask = mask
            best_count = count
            best_error = (
                consensus_error
            )

            estimated_trials = (
                required_ransac_trials(
                    confidence,
                    count / n,
                    sample_size,
                )
            )

            required_trials = min(
                required_trials,
                max(
                    estimated_trials,
                    trials,
                ),
            )

    if best_mask is None:
        raise RuntimeError(
            "No valid RANSAC "
            "hypothesis was found"
        )

    beta_final = weighted_least_squares(
        Phi[best_mask],
        y[best_mask],
    )

    final_consensus_fraction = (
        best_count / n
    )

    theoretical_trials_from_final_consensus = (
        required_ransac_trials(
            confidence,
            final_consensus_fraction,
            sample_size,
        )
    )

    info = {
        "trials": trials,
        "consensus_fraction": (
            final_consensus_fraction
        ),
        "best_count": best_count,
        "sample_size": sample_size,
        "hit_max_trials": (
            trials >= max_trials
        ),
        "theoretical_trials_from_final_consensus": (
            theoretical_trials_from_final_consensus
        ),
    }

    if return_info:
        return beta_final, info

    return beta_final


### GNC-GM

Article 1 settings:

- \(\bar c^2=\chi^2_{0.99,1}\);
- division factor \(1.4\);
- at most \(100\) continuation steps.


In [ ]:
def fit_gnc_gm(
    Phi,
    y,
    sigma,
    division_factor=1.4,
    max_iter=100,
):
    beta = weighted_least_squares(
        Phi,
        y,
    )

    c2 = chi2.ppf(
        0.99,
        df=1,
    )

    residuals = (
        normalized_squared_residuals(
            beta,
            Phi,
            y,
            sigma,
        )
    )

    mu = (
        2.0
        * residuals.max()
        / c2
    )

    if mu < 1.0:
        return beta

    for _ in range(max_iter):
        weights = (
            (c2 * mu)
            / (
                residuals
                + c2 * mu
            )
        ) ** 2

        beta = (
            weighted_least_squares(
                Phi,
                y,
                weights,
            )
        )

        residuals = (
            normalized_squared_residuals(
                beta,
                Phi,
                y,
                sigma,
            )
        )

        mu /= division_factor

        if mu < 1.0:
            break

    return beta


### GNC-TLS

Article 1 settings:

- \(\bar c^2=\chi^2_{0.99,1}\);
- multiplication factor \(1.4\);
- tolerance \(10^{-5}\);
- at most \(100\) iterations.


In [ ]:
def fit_gnc_tls(
    Phi,
    y,
    sigma,
    multiplication_factor=1.4,
    tol=1e-5,
    max_iter=100,
):
    beta = weighted_least_squares(
        Phi,
        y,
    )

    c2 = chi2.ppf(
        0.99,
        df=1,
    )

    residuals = (
        normalized_squared_residuals(
            beta,
            Phi,
            y,
            sigma,
        )
    )

    denominator = (
        2.0 * residuals.max()
        - c2
    )

    if denominator <= 1e-12:
        return beta

    mu = c2 / denominator

    previous_objective = (
        residuals.sum()
    )

    for _ in range(max_iter):
        upper = (
            (mu + 1.0)
            / mu
        ) * c2

        lower = (
            mu
            / (mu + 1.0)
        ) * c2

        weights = np.empty_like(
            residuals
        )

        high = residuals >= upper
        low = residuals <= lower
        middle = ~(high | low)

        weights[high] = 0.0
        weights[low] = 1.0

        weights[middle] = (
            np.sqrt(
                c2
                * mu
                * (mu + 1.0)
                / np.maximum(
                    residuals[middle],
                    1e-12,
                )
            )
            - mu
        )

        weights = np.clip(
            weights,
            0.0,
            1.0,
        )

        beta = (
            weighted_least_squares(
                Phi,
                y,
                weights,
            )
        )

        residuals = (
            normalized_squared_residuals(
                beta,
                Phi,
                y,
                sigma,
            )
        )

        objective = np.sum(
            weights * residuals
        )

        relative_change = (
            abs(
                objective
                - previous_objective
            )
            / max(
                abs(
                    previous_objective
                ),
                1e-12,
            )
        )

        if (
            relative_change
            <= tol
        ):
            break

        previous_objective = (
            objective
        )

        mu *= (
            multiplication_factor
        )

    return beta


### ASOR

Article 1 settings:

\[
a_0=0.5,\qquad
b_0=10^4,\qquad
A=10^4,\qquad
B=10^3,\qquad
\theta=0.5,
\]

with tolerance \(10^{-5}\) and at most \(100\) iterations.


In [ ]:
def fit_asor(
    Phi,
    y,
    sigma,
    a0=0.5,
    b0=10_000.0,
    prior_a=10_000.0,
    prior_b=1_000.0,
    theta=0.5,
    tol=1e-5,
    max_iter=100,
):
    weights = np.ones(
        len(y)
    )

    b = b0
    alpha = a0 + 0.5

    zeta = (
        (1.0 / theta - 1.0)
        * gamma(alpha)
        / gamma(a0)
    )

    previous_objective = None

    beta = (
        weighted_least_squares(
            Phi,
            y,
        )
    )

    for _ in range(max_iter):
        beta = (
            weighted_least_squares(
                Phi,
                y,
                weights,
            )
        )

        residuals = (
            normalized_squared_residuals(
                beta,
                Phi,
                y,
                sigma,
            )
        )

        beta_i = (
            0.5 * residuals
            + b
        )

        log_term = (
            np.log(zeta)
            + a0
            * np.log(
                max(
                    b,
                    1e-12,
                )
            )
            - alpha
            * np.log(
                np.maximum(
                    beta_i,
                    1e-12,
                )
            )
            + 0.5
            * residuals
        )

        omega = expit(
            -log_term
        )

        numerator = (
            prior_a
            - 1.0
            + np.sum(
                a0
                * (
                    1.0
                    - omega
                )
            )
        )

        denominator = (
            prior_b
            + np.sum(
                (
                    1.0
                    - omega
                )
                * alpha
                / beta_i
            )
        )

        b = (
            numerator
            / denominator
        )

        new_weights = (
            omega
            + (
                1.0
                - omega
            )
            * alpha
            / beta_i
        )

        objective = np.sum(
            new_weights
            * residuals
        )

        if (
            previous_objective
            is not None
        ):
            relative_change = (
                abs(
                    objective
                    - previous_objective
                )
                / max(
                    abs(
                        previous_objective
                    ),
                    1e-12,
                )
            )

            if (
                relative_change
                <= tol
            ):
                weights = (
                    new_weights
                )
                break

        previous_objective = (
            objective
        )

        weights = new_weights

    return (
        weighted_least_squares(
            Phi,
            y,
            weights,
        )
    )


In [ ]:
FITTERS = {
    "OLS": fit_ols,
    "Huber": fit_huber,
    "RANSAC": fit_ransac,
    "GNC-GM": fit_gnc_gm,
    "GNC-TLS": fit_gnc_tls,
    "ASOR": fit_asor,
}


## 5. Controlled contamination

The Gaussian and uniform replacement mechanisms follow the Article 1 generator:

\[
\mu_{\mathrm{out}}
=
\kappa_\mu\sigma_{\mathrm{in}},
\qquad
\sigma_{\mathrm{out}}^2
=
\kappa_\sigma\sigma_{\mathrm{in}}^2.
\]

The nominal measurement error is **replaced**, not supplemented, at corrupted indices.

The coherent competing-line case from Article 1 is generalized to a coherent competing curve. To keep the alternative relationship identical across Monte Carlo realizations, it is defined relative to the fixed domain reference \(x=0\):

\[
f_{\mathrm{alt}}(x)
=
f(0)+5\,[f(x)-f(0)].
\]

For polynomial models this leaves the constant term unchanged and multiplies all nonconstant structure by five, directly mirroring the five-times-slope construction used for the competing line.


In [ ]:
def competing_curve(
    x,
    model_name,
    amplification=5.0,
):
    x = np.asarray(
        x,
        dtype=float,
    )

    nominal = true_curve(
        x,
        model_name,
    )

    reference_level = float(
        true_curve(
            np.array([0.0]),
            model_name,
        )[0]
    )

    return (
        reference_level
        + amplification
        * (
            nominal
            - reference_level
        )
    )


def generate_contaminated_data(
    model_name,
    n,
    outlier_ratio,
    variance_in,
    contamination_type,
    seed,
):
    rng = np.random.default_rng(
        seed
    )

    x = rng.uniform(
        X_MIN,
        X_MAX,
        n,
    )

    nominal_curve = true_curve(
        x,
        model_name,
    )

    sigma_in = np.sqrt(
        variance_in
    )

    y = (
        nominal_curve
        + rng.normal(
            0.0,
            sigma_in,
            size=n,
        )
    )

    n_outliers = round(
        outlier_ratio * n
    )

    outlier_mask = np.zeros(
        n,
        dtype=bool,
    )

    if n_outliers == 0:
        return (
            x,
            y,
            nominal_curve,
            outlier_mask,
        )

    selected = rng.choice(
        n,
        n_outliers,
        replace=False,
    )

    outlier_mask[selected] = True

    if (
        contamination_type
        == "zero_mean_gaussian"
    ):
        mean_out = 0.0
        variance_out = (
            VARIANCE_RATIO
            * variance_in
        )

        outlier_noise = rng.normal(
            mean_out,
            np.sqrt(
                variance_out
            ),
            n_outliers,
        )

        y[selected] = (
            nominal_curve[selected]
            + outlier_noise
        )

    elif (
        contamination_type
        == "biased_gaussian"
    ):
        mean_out = (
            BIASED_MEAN_SHIFT_SIGMA
            * sigma_in
        )

        variance_out = (
            VARIANCE_RATIO
            * variance_in
        )

        outlier_noise = rng.normal(
            mean_out,
            np.sqrt(
                variance_out
            ),
            n_outliers,
        )

        y[selected] = (
            nominal_curve[selected]
            + outlier_noise
        )

    elif (
        contamination_type
        == "biased_uniform"
    ):
        mean_out = (
            BIASED_MEAN_SHIFT_SIGMA
            * sigma_in
        )

        half_width = np.sqrt(
            3.0
            * VARIANCE_RATIO
            * variance_in
        )

        outlier_noise = rng.uniform(
            mean_out - half_width,
            mean_out + half_width,
            n_outliers,
        )

        y[selected] = (
            nominal_curve[selected]
            + outlier_noise
        )

    elif (
        contamination_type
        == "competing_curve"
    ):
        alternative = (
            competing_curve(
                x,
                model_name,
                amplification=5.0,
            )
        )

        y[selected] = (
            alternative[selected]
            + rng.normal(
                0.0,
                sigma_in,
                n_outliers,
            )
        )

    else:
        raise ValueError(
            "Unknown contamination type"
        )

    return (
        x,
        y,
        nominal_curve,
        outlier_mask,
    )


## 6. Prediction error and one-fit wrapper

Prediction RMSE is evaluated against the noiseless nominal relationship on a dense grid:

\[
\operatorname{RMSE}_{\mathrm{pred}}
=
\sqrt{
\frac{1}{J}
\sum_{j=1}^{J}
[
\widehat f(x_j)-f(x_j)
]^2
}.
\]

For the noise sweep,

\[
\operatorname{NRMSE}
=
\frac{
\operatorname{RMSE}_{\mathrm{pred}}
}{
\sigma_{\mathrm{in}}
}.
\]


In [ ]:
def prediction_rmse(
    beta,
    model_name,
    x_eval,
):
    Phi_eval = design_matrix(
        x_eval,
        model_name,
    )

    y_true_eval = true_curve(
        x_eval,
        model_name,
    )

    y_hat_eval = predict(
        beta,
        Phi_eval,
    )

    return np.sqrt(
        np.mean(
            (
                y_hat_eval
                - y_true_eval
            ) ** 2
        )
    )


def fit_method(
    method,
    Phi,
    y,
    sigma,
    seed,
):
    diagnostics = {
        "ransac_trials": np.nan,
        "ransac_consensus_fraction": np.nan,
        "ransac_hit_max_trials": False,
        "ransac_theoretical_trials_final": np.nan,
    }

    start = time.perf_counter()

    if method == "OLS":
        beta = fit_ols(
            Phi,
            y,
        )

    elif method == "Huber":
        beta = fit_huber(
            Phi,
            y,
            sigma,
        )

    elif method == "RANSAC":
        beta, info = fit_ransac(
            Phi,
            y,
            sigma,
            threshold=3.0,
            confidence=0.999,
            max_trials=10_000,
            seed=seed,
            return_info=True,
        )

        diagnostics.update({
            "ransac_trials": (
                info["trials"]
            ),
            "ransac_consensus_fraction": (
                info[
                    "consensus_fraction"
                ]
            ),
            "ransac_hit_max_trials": (
                info[
                    "hit_max_trials"
                ]
            ),
            "ransac_theoretical_trials_final": (
                info[
                    "theoretical_trials_from_final_consensus"
                ]
            ),
        })

    elif method == "GNC-GM":
        beta = fit_gnc_gm(
            Phi,
            y,
            sigma,
        )

    elif method == "GNC-TLS":
        beta = fit_gnc_tls(
            Phi,
            y,
            sigma,
        )

    elif method == "ASOR":
        beta = fit_asor(
            Phi,
            y,
            sigma,
        )

    else:
        raise ValueError(
            f"Unknown method: {method}"
        )

    runtime = (
        time.perf_counter()
        - start
    )

    return (
        beta,
        runtime,
        diagnostics,
    )


## 7. Sanity check

This is deliberately small. Run it before the complete benchmark.


In [ ]:
x_eval = np.linspace(
    X_MIN,
    X_MAX,
    1000,
)

sanity_rows = []

for model_index, model_name in enumerate(
    MODEL_NAMES
):
    seed = (
        MASTER_SEED
        + 10000 * model_index
    )

    x, y, _, _ = (
        generate_contaminated_data(
            model_name=model_name,
            n=100,
            outlier_ratio=0.30,
            variance_in=1000.0,
            contamination_type=(
                "biased_gaussian"
            ),
            seed=seed,
        )
    )

    Phi = design_matrix(
        x,
        model_name,
    )

    sigma = np.sqrt(1000.0)

    for method_index, method in enumerate(
        METHOD_NAMES
    ):
        beta, runtime, diagnostics = fit_method(
            method,
            Phi,
            y,
            sigma,
            seed=(
                seed
                + method_index
            ),
        )

        rmse = prediction_rmse(
            beta,
            model_name,
            x_eval,
        )

        sanity_rows.append({
            "model": model_name,
            "method": method,
            "RMSE_pred": rmse,
            "runtime_s": runtime,
            **diagnostics,
        })

sanity = pd.DataFrame(
    sanity_rows
)

sanity


## 8. Monte Carlo engine

Within each Monte Carlo realization, **all six estimators receive exactly the same generated dataset**.


In [ ]:
def run_condition(
    model_name,
    n,
    outlier_ratio,
    variance_in,
    contamination_type,
    mc_runs,
    seed_offset=0,
):
    sigma = np.sqrt(
        variance_in
    )

    x_eval = np.linspace(
        X_MIN,
        X_MAX,
        1200,
    )

    rows = []

    for mc in range(mc_runs):
        seed = (
            MASTER_SEED
            + seed_offset
            + 100003 * mc
            + 1009 * n
            + int(
                1000
                * outlier_ratio
            )
        )

        x, y, _, outlier_mask = (
            generate_contaminated_data(
                model_name=model_name,
                n=n,
                outlier_ratio=(
                    outlier_ratio
                ),
                variance_in=(
                    variance_in
                ),
                contamination_type=(
                    contamination_type
                ),
                seed=seed,
            )
        )

        Phi = design_matrix(
            x,
            model_name,
        )

        for method_index, method in enumerate(
            METHOD_NAMES
        ):
            beta, runtime, diagnostics = fit_method(
                method,
                Phi,
                y,
                sigma,
                seed=(
                    seed
                    + method_index
                ),
            )

            rmse = prediction_rmse(
                beta,
                model_name,
                x_eval,
            )

            rows.append({
                "model": model_name,
                "method": method,
                "contamination_type": (
                    contamination_type
                ),
                "outlier_ratio": (
                    outlier_ratio
                ),
                "N": n,
                "variance_in": (
                    variance_in
                ),
                "sigma_in": sigma,
                "mc": mc,
                "RMSE_pred": rmse,
                "NRMSE": (
                    rmse / sigma
                ),
                "runtime_s": runtime,
                "actual_outliers": int(
                    outlier_mask.sum()
                ),
                **diagnostics,
            })

    return pd.DataFrame(
        rows
    )


def summarize_metric(
    data,
    group_columns,
    metric,
):
    return (
        data
        .groupby(
            group_columns
        )[metric]
        .agg(
            median="median",
            mean="mean",
            std="std",
            q25=lambda s: (
                s.quantile(0.25)
            ),
            q75=lambda s: (
                s.quantile(0.75)
            ),
            count="count",
        )
        .reset_index()
    )


## 9. Contamination sweep

Main configuration:

\[
N=100,\qquad
\sigma_{\mathrm{in}}^2=1000.
\]

The four contamination models are:

1. zero-mean Gaussian replacement, \(\kappa_\sigma=10\);
2. biased Gaussian replacement, \(\kappa_\sigma=10,\ \kappa_\mu=3\);
3. biased uniform replacement with the same mean and variance as case 2;
4. a coherent competing curve.


In [ ]:
contamination_frames = []

for contamination_index, contamination_type in enumerate(
    CONTAMINATION_TYPES
):
    for model_index, model_name in enumerate(
        MODEL_NAMES
    ):
        for ratio in CONTAMINATION_RATIOS:
            result = run_condition(
                model_name=model_name,
                n=BASELINE_N,
                outlier_ratio=float(ratio),
                variance_in=(
                    BASELINE_VARIANCE
                ),
                contamination_type=(
                    contamination_type
                ),
                mc_runs=MC_RUNS,
                seed_offset=(
                    10_000_000
                    * contamination_index
                    + 1_000_000
                    * model_index
                ),
            )

            contamination_frames.append(
                result
            )

contamination_results = pd.concat(
    contamination_frames,
    ignore_index=True,
)

contamination_results.to_csv(
    RESULTS_DIR
    / "contamination_sweep.csv",
    index=False,
)

contamination_summary = (
    summarize_metric(
        contamination_results,
        [
            "model",
            "contamination_type",
            "outlier_ratio",
            "method",
        ],
        "NRMSE",
    )
)

contamination_runtime_summary = (
    summarize_metric(
        contamination_results,
        [
            "model",
            "contamination_type",
            "outlier_ratio",
            "method",
        ],
        "runtime_s",
    )
)

contamination_summary.head()


In [ ]:
markers = [
    "o", "s", "^", "D", "P", "X"
]

for model_name in MODEL_NAMES:
    for contamination_type in CONTAMINATION_TYPES:
        subset = contamination_summary[
            (
                contamination_summary[
                    "model"
                ]
                == model_name
            )
            & (
                contamination_summary[
                    "contamination_type"
                ]
                == contamination_type
            )
        ]

        plt.figure(
            figsize=(7.2, 4.8)
        )

        for marker, method in zip(
            markers,
            METHOD_NAMES,
        ):
            dm = subset[
                subset["method"]
                == method
            ].sort_values(
                "outlier_ratio"
            )

            x_plot = (
                100.0
                * dm["outlier_ratio"].to_numpy()
            )
            y_plot = dm["median"].to_numpy()
            yerr = np.vstack([
                y_plot
                - dm["q25"].to_numpy(),
                dm["q75"].to_numpy()
                - y_plot,
            ])

            plt.errorbar(
                x_plot,
                y_plot,
                yerr=yerr,
                marker=marker,
                linewidth=1.5,
                capsize=2,
                label=method,
            )

        plt.yscale("log")
        plt.xlabel(
            "Contamination ratio (%)"
        )
        plt.ylabel(
            "Prediction NRMSE"
        )
        plt.title(
            f"{model_name}: "
            f"{contamination_type.replace('_', ' ')}"
        )
        plt.grid(
            alpha=0.25
        )
        plt.legend()
        plt.tight_layout()

        plt.savefig(
            FIGURES_DIR
            / (
                f"contamination_"
                f"{model_name}_"
                f"{contamination_type}.png"
            ),
            dpi=220,
        )

        plt.show()


# Runtime figures for the biased Gaussian benchmark.
for model_name in MODEL_NAMES:
    subset = contamination_runtime_summary[
        (
            contamination_runtime_summary[
                "model"
            ]
            == model_name
        )
        & (
            contamination_runtime_summary[
                "contamination_type"
            ]
            == "biased_gaussian"
        )
    ]

    plt.figure(
        figsize=(7.2, 4.8)
    )

    for marker, method in zip(
        markers,
        METHOD_NAMES,
    ):
        dm = subset[
            subset["method"]
            == method
        ].sort_values(
            "outlier_ratio"
        )

        x_plot = (
            100.0
            * dm["outlier_ratio"].to_numpy()
        )
        y_plot = dm["median"].to_numpy()
        yerr = np.vstack([
            y_plot
            - dm["q25"].to_numpy(),
            dm["q75"].to_numpy()
            - y_plot,
        ])

        plt.errorbar(
            x_plot,
            y_plot,
            yerr=yerr,
            marker=marker,
            linewidth=1.5,
            capsize=2,
            label=method,
        )

    plt.yscale("log")
    plt.xlabel(
        "Contamination ratio (%)"
    )
    plt.ylabel(
        "Runtime (s)"
    )
    plt.title(
        f"Runtime under biased Gaussian contamination: {model_name}"
    )
    plt.grid(
        alpha=0.25
    )
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        FIGURES_DIR
        / f"runtime_biased_gaussian_{model_name}.png",
        dpi=220,
    )

    plt.show()


## 10. Sample-size scaling

As in Article 1:

- \(50\%\) biased Gaussian contamination;
- \(\sigma_{\mathrm{in}}^2=1000\);
- \(\kappa_\sigma=10\);
- \(\kappa_\mu=3\);
- \(N\in\{50,100,200,500,1000,2000,5000\}\).


In [ ]:
sample_frames = []

for model_index, model_name in enumerate(
    MODEL_NAMES
):
    for n in SAMPLE_SIZES:
        result = run_condition(
            model_name=model_name,
            n=n,
            outlier_ratio=0.50,
            variance_in=1000.0,
            contamination_type=(
                "biased_gaussian"
            ),
            mc_runs=MC_RUNS,
            seed_offset=(
                30_000_000
                + 1_000_000
                * model_index
            ),
        )

        sample_frames.append(
            result
        )

sample_results = pd.concat(
    sample_frames,
    ignore_index=True,
)

sample_results.to_csv(
    RESULTS_DIR
    / "sample_size_sweep.csv",
    index=False,
)

sample_summary = summarize_metric(
    sample_results,
    ["model", "N", "method"],
    "NRMSE",
)

sample_runtime_summary = summarize_metric(
    sample_results,
    ["model", "N", "method"],
    "runtime_s",
)

sample_summary.head()


In [ ]:
for model_name in MODEL_NAMES:
    subset = sample_summary[
        sample_summary["model"]
        == model_name
    ]

    plt.figure(
        figsize=(7.2, 4.8)
    )

    for marker, method in zip(
        markers,
        METHOD_NAMES,
    ):
        dm = subset[
            subset["method"]
            == method
        ].sort_values("N")

        x_plot = dm["N"].to_numpy()
        y_plot = dm["median"].to_numpy()
        yerr = np.vstack([
            y_plot
            - dm["q25"].to_numpy(),
            dm["q75"].to_numpy()
            - y_plot,
        ])

        plt.errorbar(
            x_plot,
            y_plot,
            yerr=yerr,
            marker=marker,
            linewidth=1.5,
            capsize=2,
            label=method,
        )

    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel(
        "Number of measurements, N"
    )
    plt.ylabel(
        "Prediction NRMSE"
    )
    plt.title(
        f"Sample-size scaling: {model_name}"
    )
    plt.grid(
        alpha=0.25
    )
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        FIGURES_DIR
        / f"sample_size_{model_name}.png",
        dpi=220,
    )

    plt.show()


for model_name in MODEL_NAMES:
    subset = sample_runtime_summary[
        sample_runtime_summary["model"]
        == model_name
    ]

    plt.figure(
        figsize=(7.2, 4.8)
    )

    for marker, method in zip(
        markers,
        METHOD_NAMES,
    ):
        dm = subset[
            subset["method"]
            == method
        ].sort_values("N")

        x_plot = dm["N"].to_numpy()
        y_plot = dm["median"].to_numpy()
        yerr = np.vstack([
            y_plot
            - dm["q25"].to_numpy(),
            dm["q75"].to_numpy()
            - y_plot,
        ])

        plt.errorbar(
            x_plot,
            y_plot,
            yerr=yerr,
            marker=marker,
            linewidth=1.5,
            capsize=2,
            label=method,
        )

    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel(
        "Number of measurements, N"
    )
    plt.ylabel(
        "Runtime (s)"
    )
    plt.title(
        f"Runtime scaling: {model_name}"
    )
    plt.grid(
        alpha=0.25
    )
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        FIGURES_DIR
        / f"runtime_sample_size_{model_name}.png",
        dpi=220,
    )

    plt.show()


## 11. Nominal-noise scaling

As in Article 1:

- \(N=500\);
- \(50\%\) biased Gaussian contamination;
- \(\kappa_\sigma=10\);
- \(\kappa_\mu=3\);
- \(\sigma_{\mathrm{in}}^2\in\{10,100,1000,10000\}\).

Each noise variance now receives a **different Monte Carlo seed stream**. This avoids obtaining numerically identical NRMSE values merely because the same standardized random realization has been rescaled.


In [ ]:
noise_frames = []

for model_index, model_name in enumerate(
    MODEL_NAMES
):
    for noise_index, variance_in in enumerate(
        NOISE_VARIANCES
    ):
        result = run_condition(
            model_name=model_name,
            n=500,
            outlier_ratio=0.50,
            variance_in=float(
                variance_in
            ),
            contamination_type=(
                "biased_gaussian"
            ),
            mc_runs=MC_RUNS,
            seed_offset=(
                50_000_000
                + 1_000_000
                * model_index
                + 100_000
                * noise_index
            ),
        )

        noise_frames.append(
            result
        )

noise_results = pd.concat(
    noise_frames,
    ignore_index=True,
)

noise_results.to_csv(
    RESULTS_DIR
    / "noise_sweep.csv",
    index=False,
)

noise_summary = summarize_metric(
    noise_results,
    [
        "model",
        "variance_in",
        "method",
    ],
    "NRMSE",
)

noise_summary.head()


In [ ]:
for model_name in MODEL_NAMES:
    subset = noise_summary[
        noise_summary["model"]
        == model_name
    ]

    plt.figure(
        figsize=(7.2, 4.8)
    )

    for marker, method in zip(
        markers,
        METHOD_NAMES,
    ):
        dm = subset[
            subset["method"]
            == method
        ].sort_values(
            "variance_in"
        )

        x_plot = dm["variance_in"].to_numpy()
        y_plot = dm["median"].to_numpy()
        yerr = np.vstack([
            y_plot
            - dm["q25"].to_numpy(),
            dm["q75"].to_numpy()
            - y_plot,
        ])

        plt.errorbar(
            x_plot,
            y_plot,
            yerr=yerr,
            marker=marker,
            linewidth=1.5,
            capsize=2,
            label=method,
        )

    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel(
        r"Nominal noise variance, $\sigma_{\rm in}^2$"
    )
    plt.ylabel(
        "Prediction NRMSE"
    )
    plt.title(
        f"Noise scaling: {model_name}"
    )
    plt.grid(
        alpha=0.25
    )
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        FIGURES_DIR
        / f"noise_scaling_{model_name}.png",
        dpi=220,
    )

    plt.show()


## 12. RANSAC minimal-sample burden

For a model with \(s\) fitted coefficients and inlier fraction \(w\),

\[
K
=
\frac{
\log(1-p)
}{
\log(1-w^s)
}.
\]

The theoretical plot isolates the combinatorial effect of increasing model dimension.

The empirical contamination results also record:

- `ransac_trials`;
- `ransac_consensus_fraction`;
- `ransac_hit_max_trials`;
- `ransac_theoretical_trials_final`.

This makes it possible to distinguish a statistical failure from a case in which the imposed \(10{,}000\)-hypothesis budget was exhausted.


In [ ]:
ransac_rows = []

for model_name in MODEL_NAMES:
    sample_size = (
        number_of_coefficients(
            model_name
        )
    )

    for ratio in CONTAMINATION_RATIOS_FULL:
        inlier_fraction = (
            1.0 - ratio
        )

        trials = (
            required_ransac_trials(
                confidence=0.999,
                inlier_fraction=(
                    inlier_fraction
                ),
                sample_size=(
                    sample_size
                ),
            )
        )

        ransac_rows.append({
            "model": model_name,
            "sample_size": sample_size,
            "outlier_ratio": ratio,
            "required_trials": trials,
        })

ransac_theory = pd.DataFrame(
    ransac_rows
)

ransac_theory.to_csv(
    RESULTS_DIR
    / "ransac_theoretical_trials.csv",
    index=False,
)

ransac_theory


In [ ]:
plt.figure(
    figsize=(7.2, 4.8)
)

for marker, model_name in zip(
    ["o", "s", "^"],
    MODEL_NAMES,
):
    dm = ransac_theory[
        ransac_theory["model"]
        == model_name
    ]

    plt.plot(
        100.0
        * dm["outlier_ratio"],
        dm["required_trials"],
        marker=marker,
        linewidth=1.6,
        label=(
            f"{model_name} "
            f"(s={number_of_coefficients(model_name)})"
        ),
    )

plt.yscale("log")
plt.xlabel(
    "Contamination ratio (%)"
)
plt.ylabel(
    "Required hypotheses "
    "for confidence 0.999"
)
plt.title(
    "RANSAC sampling burden"
)
plt.grid(
    alpha=0.25
)
plt.legend()
plt.tight_layout()

plt.savefig(
    FIGURES_DIR
    / "ransac_sampling_burden.png",
    dpi=220,
)

plt.show()


### Empirical RANSAC budget diagnostics

The next table summarizes the measured RANSAC behavior from the contamination sweep. A nonzero `hit_cap_fraction` means that some Monte Carlo realizations exhausted the \(10{,}000\)-hypothesis budget.


In [ ]:
ransac_diagnostics = (
    contamination_results[
        contamination_results["method"]
        == "RANSAC"
    ]
    .groupby(
        [
            "model",
            "contamination_type",
            "outlier_ratio",
        ],
        as_index=False,
    )
    .agg(
        median_trials=(
            "ransac_trials",
            "median",
        ),
        median_consensus_fraction=(
            "ransac_consensus_fraction",
            "median",
        ),
        hit_cap_fraction=(
            "ransac_hit_max_trials",
            "mean",
        ),
        median_runtime_s=(
            "runtime_s",
            "median",
        ),
        median_NRMSE=(
            "NRMSE",
            "median",
        ),
    )
)

ransac_diagnostics.to_csv(
    RESULTS_DIR
    / "ransac_empirical_diagnostics.csv",
    index=False,
)

ransac_diagnostics


## 13. Common-Truth Model-Richness Experiment

The main benchmark uses a different nominal curve for each model class. That is useful for studying robustness within each representation, but it does not completely isolate the effect of increasing the number of coefficients.

This controlled experiment therefore generates **one quadratic truth** and fits all three bases to the **same \(x\), \(y\), outlier locations, and noise realization**:

\[
\text{quadratic }(3\text{ coefficients}),\qquad
\text{cubic }(4),\qquad
\text{quadratic B-spline }(8).
\]

Because a cubic basis and a quadratic B-spline basis can both represent a quadratic function, differences now reflect model richness and estimator behavior rather than a different underlying relationship.

The contamination model is the Article 1 biased Gaussian replacement case with \(\kappa_\sigma=10\) and \(\kappa_\mu=3\).


In [ ]:
COMMON_TRUTH_BETA = np.array([
    20.0,
    260.0,
    120.0,
])


def shared_quadratic_truth(x):
    return (
        polynomial_design(
            x,
            degree=2,
        )
        @ COMMON_TRUTH_BETA
    )


def generate_common_truth_data(
    n,
    outlier_ratio,
    variance_in,
    seed,
):
    rng = np.random.default_rng(
        seed
    )

    x = rng.uniform(
        X_MIN,
        X_MAX,
        n,
    )

    y_true = shared_quadratic_truth(
        x
    )

    sigma = np.sqrt(
        variance_in
    )

    y = (
        y_true
        + rng.normal(
            0.0,
            sigma,
            n,
        )
    )

    n_outliers = round(
        outlier_ratio * n
    )

    outlier_mask = np.zeros(
        n,
        dtype=bool,
    )

    if n_outliers > 0:
        selected = rng.choice(
            n,
            n_outliers,
            replace=False,
        )

        outlier_mask[selected] = True

        mean_out = (
            BIASED_MEAN_SHIFT_SIGMA
            * sigma
        )

        variance_out = (
            VARIANCE_RATIO
            * variance_in
        )

        outlier_noise = rng.normal(
            mean_out,
            np.sqrt(
                variance_out
            ),
            n_outliers,
        )

        y[selected] = (
            y_true[selected]
            + outlier_noise
        )

    return (
        x,
        y,
        y_true,
        outlier_mask,
    )


def prediction_rmse_against_shared_truth(
    beta,
    model_name,
    x_eval,
):
    Phi_eval = design_matrix(
        x_eval,
        model_name,
    )

    y_hat = Phi_eval @ beta
    y_true = shared_quadratic_truth(
        x_eval
    )

    return np.sqrt(
        np.mean(
            (
                y_hat
                - y_true
            ) ** 2
        )
    )


In [ ]:
common_truth_ratios = (
    CONTAMINATION_RATIOS_FULL
    if RUN_PROFILE == "full"
    else np.array([
        0.0,
        0.30,
        0.50,
    ])
)

common_truth_rows = []
x_eval_common = np.linspace(
    X_MIN,
    X_MAX,
    1200,
)

for ratio in common_truth_ratios:
    for mc in range(MC_RUNS):
        seed = (
            MASTER_SEED
            + 70_000_000
            + 100003 * mc
            + int(
                1000 * ratio
            )
        )

        x, y, _, outlier_mask = (
            generate_common_truth_data(
                n=BASELINE_N,
                outlier_ratio=float(
                    ratio
                ),
                variance_in=(
                    BASELINE_VARIANCE
                ),
                seed=seed,
            )
        )

        sigma = np.sqrt(
            BASELINE_VARIANCE
        )

        for model_index, model_name in enumerate(
            MODEL_NAMES
        ):
            Phi = design_matrix(
                x,
                model_name,
            )

            for method_index, method in enumerate(
                METHOD_NAMES
            ):
                beta, runtime, diagnostics = fit_method(
                    method,
                    Phi,
                    y,
                    sigma,
                    seed=(
                        seed
                        + 1000
                        * model_index
                        + method_index
                    ),
                )

                rmse = (
                    prediction_rmse_against_shared_truth(
                        beta,
                        model_name,
                        x_eval_common,
                    )
                )

                common_truth_rows.append({
                    "model": model_name,
                    "method": method,
                    "outlier_ratio": float(
                        ratio
                    ),
                    "N": BASELINE_N,
                    "variance_in": (
                        BASELINE_VARIANCE
                    ),
                    "mc": mc,
                    "RMSE_pred": rmse,
                    "NRMSE": (
                        rmse / sigma
                    ),
                    "runtime_s": runtime,
                    "actual_outliers": int(
                        outlier_mask.sum()
                    ),
                    **diagnostics,
                })

common_truth_results = pd.DataFrame(
    common_truth_rows
)

common_truth_results.to_csv(
    RESULTS_DIR
    / "common_truth_model_richness.csv",
    index=False,
)

common_truth_summary = summarize_metric(
    common_truth_results,
    [
        "model",
        "outlier_ratio",
        "method",
    ],
    "NRMSE",
)

common_truth_summary.head()


In [ ]:
for model_name in MODEL_NAMES:
    subset = common_truth_summary[
        common_truth_summary["model"]
        == model_name
    ]

    plt.figure(
        figsize=(7.2, 4.8)
    )

    for marker, method in zip(
        markers,
        METHOD_NAMES,
    ):
        dm = subset[
            subset["method"]
            == method
        ].sort_values(
            "outlier_ratio"
        )

        x_plot = (
            100.0
            * dm["outlier_ratio"].to_numpy()
        )
        y_plot = dm["median"].to_numpy()
        yerr = np.vstack([
            y_plot
            - dm["q25"].to_numpy(),
            dm["q75"].to_numpy()
            - y_plot,
        ])

        plt.errorbar(
            x_plot,
            y_plot,
            yerr=yerr,
            marker=marker,
            linewidth=1.5,
            capsize=2,
            label=method,
        )

    plt.yscale("log")
    plt.xlabel(
        "Contamination ratio (%)"
    )
    plt.ylabel(
        "Prediction NRMSE"
    )
    plt.title(
        f"Common quadratic truth: {model_name} fit"
    )
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        FIGURES_DIR
        / f"common_truth_{model_name}.png",
        dpi=220,
    )

    plt.show()


## 14. Local-Support Experiment

This experiment targets a different question: when contamination is localized in \(x\), does a local spline basis limit how far the deformation spreads?

The nominal truth is again quadratic so that both a cubic polynomial basis and the quadratic B-spline basis can represent it.

Crucially, **not every measurement inside the contaminated interval is corrupted**. By default, only \(50\%\) of the measurements in \([-20,20]\) are replaced. The remaining measurements in the same interval stay nominal, so the local relationship remains identifiable.

Both OLS and ASOR are shown to separate the effect of the basis from the effect of robust weighting.


In [ ]:
def generate_localized_contamination(
    n=500,
    variance_in=1000.0,
    x_band=(-20.0, 20.0),
    local_contamination_fraction=0.50,
    mean_shift_sigma=3.0,
    variance_ratio=10.0,
    seed=MASTER_SEED + 777,
):
    rng = np.random.default_rng(
        seed
    )

    x = rng.uniform(
        X_MIN,
        X_MAX,
        n,
    )

    y_true = shared_quadratic_truth(
        x
    )

    sigma = np.sqrt(
        variance_in
    )

    y = (
        y_true
        + rng.normal(
            0.0,
            sigma,
            n,
        )
    )

    local_candidates = np.flatnonzero(
        (x >= x_band[0])
        & (x <= x_band[1])
    )

    n_local_outliers = round(
        local_contamination_fraction
        * len(local_candidates)
    )

    contaminated = np.zeros(
        n,
        dtype=bool,
    )

    if n_local_outliers > 0:
        selected = rng.choice(
            local_candidates,
            n_local_outliers,
            replace=False,
        )

        contaminated[selected] = True

        outlier_noise = rng.normal(
            mean_shift_sigma * sigma,
            np.sqrt(
                variance_ratio
                * variance_in
            ),
            n_local_outliers,
        )

        y[selected] = (
            y_true[selected]
            + outlier_noise
        )

    return (
        x,
        y,
        y_true,
        contaminated,
        local_candidates,
    )


x_local, y_local, _, local_mask, local_candidates = (
    generate_localized_contamination()
)

x_eval_local = np.linspace(
    X_MIN,
    X_MAX,
    1400,
)

truth_eval_local = (
    shared_quadratic_truth(
        x_eval_local
    )
)

local_rows = []
local_predictions = {}

for model_name in [
    "cubic",
    "bspline",
]:
    Phi = design_matrix(
        x_local,
        model_name,
    )

    Phi_eval = design_matrix(
        x_eval_local,
        model_name,
    )

    for method in [
        "OLS",
        "ASOR",
    ]:
        beta, runtime, diagnostics = fit_method(
            method,
            Phi,
            y_local,
            np.sqrt(1000.0),
            seed=MASTER_SEED + 888,
        )

        prediction = (
            Phi_eval @ beta
        )

        local_predictions[
            (model_name, method)
        ] = prediction

        near = (
            (x_eval_local >= -20.0)
            & (x_eval_local <= 20.0)
        )

        far = ~near

        local_rows.append({
            "model": model_name,
            "method": method,
            "RMSE_all": np.sqrt(
                np.mean(
                    (
                        prediction
                        - truth_eval_local
                    ) ** 2
                )
            ),
            "RMSE_local_region": np.sqrt(
                np.mean(
                    (
                        prediction[near]
                        - truth_eval_local[near]
                    ) ** 2
                )
            ),
            "RMSE_away_from_region": np.sqrt(
                np.mean(
                    (
                        prediction[far]
                        - truth_eval_local[far]
                    ) ** 2
                )
            ),
            "runtime_s": runtime,
            "local_candidate_count": int(
                len(local_candidates)
            ),
            "local_outlier_count": int(
                local_mask.sum()
            ),
            "local_contamination_fraction_realized": (
                local_mask.sum()
                / max(
                    len(local_candidates),
                    1,
                )
            ),
        })

local_support_results = pd.DataFrame(
    local_rows
)

local_support_results.to_csv(
    RESULTS_DIR
    / "local_support_experiment.csv",
    index=False,
)

local_support_results


In [ ]:
plt.figure(
    figsize=(7.4, 4.9)
)

plt.scatter(
    x_local[~local_mask],
    y_local[~local_mask],
    s=14,
    facecolors="none",
    label="Nominal observations",
)

plt.scatter(
    x_local[local_mask],
    y_local[local_mask],
    s=24,
    marker="x",
    label="Localized outliers",
)

plt.plot(
    x_eval_local,
    truth_eval_local,
    linewidth=2.2,
    label="True relationship",
)

for (
    model_name,
    method,
), prediction in local_predictions.items():
    plt.plot(
        x_eval_local,
        prediction,
        linewidth=1.5,
        label=(
            f"{method} — {model_name}"
        ),
    )

plt.xlabel("x")
plt.ylabel("y")
plt.title(
    "Localized partial contamination: "
    "global polynomial vs local spline"
)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

plt.savefig(
    FIGURES_DIR
    / "local_support_comparison.png",
    dpi=220,
)

plt.show()


## 15. Runtime Summaries


In [ ]:
runtime_summary = (
    contamination_results
    .groupby(
        [
            "model",
            "contamination_type",
            "outlier_ratio",
            "method",
        ],
        as_index=False,
    )
    .agg(
        median_runtime_s=(
            "runtime_s",
            "median",
        ),
        median_NRMSE=(
            "NRMSE",
            "median",
        ),
    )
)

runtime_summary.to_csv(
    RESULTS_DIR
    / "runtime_summary.csv",
    index=False,
)

runtime_summary.head(20)


## 16. Article-Facing Summary Tables


In [ ]:
article_contamination_summary = (
    contamination_results
    .groupby(
        [
            "model",
            "contamination_type",
            "outlier_ratio",
            "method",
        ],
        as_index=False,
    )
    .agg(
        median_RMSE=(
            "RMSE_pred",
            "median",
        ),
        median_NRMSE=(
            "NRMSE",
            "median",
        ),
        q25_NRMSE=(
            "NRMSE",
            lambda s: (
                s.quantile(0.25)
            ),
        ),
        q75_NRMSE=(
            "NRMSE",
            lambda s: (
                s.quantile(0.75)
            ),
        ),
        median_runtime_s=(
            "runtime_s",
            "median",
        ),
        median_ransac_trials=(
            "ransac_trials",
            "median",
        ),
        ransac_hit_cap_fraction=(
            "ransac_hit_max_trials",
            "mean",
        ),
    )
)

article_contamination_summary.to_csv(
    RESULTS_DIR
    / "article_contamination_summary.csv",
    index=False,
)

article_sample_summary = (
    sample_results
    .groupby(
        [
            "model",
            "N",
            "method",
        ],
        as_index=False,
    )
    .agg(
        median_NRMSE=(
            "NRMSE",
            "median",
        ),
        median_runtime_s=(
            "runtime_s",
            "median",
        ),
        median_ransac_trials=(
            "ransac_trials",
            "median",
        ),
        ransac_hit_cap_fraction=(
            "ransac_hit_max_trials",
            "mean",
        ),
    )
)

article_sample_summary.to_csv(
    RESULTS_DIR
    / "article_sample_summary.csv",
    index=False,
)

article_noise_summary = (
    noise_results
    .groupby(
        [
            "model",
            "variance_in",
            "method",
        ],
        as_index=False,
    )
    .agg(
        median_NRMSE=(
            "NRMSE",
            "median",
        ),
        median_runtime_s=(
            "runtime_s",
            "median",
        ),
    )
)

article_noise_summary.to_csv(
    RESULTS_DIR
    / "article_noise_summary.csv",
    index=False,
)

common_truth_article_summary = (
    common_truth_results
    .groupby(
        [
            "model",
            "outlier_ratio",
            "method",
        ],
        as_index=False,
    )
    .agg(
        median_NRMSE=(
            "NRMSE",
            "median",
        ),
        q25_NRMSE=(
            "NRMSE",
            lambda s: s.quantile(0.25),
        ),
        q75_NRMSE=(
            "NRMSE",
            lambda s: s.quantile(0.75),
        ),
        median_runtime_s=(
            "runtime_s",
            "median",
        ),
        median_ransac_trials=(
            "ransac_trials",
            "median",
        ),
        ransac_hit_cap_fraction=(
            "ransac_hit_max_trials",
            "mean",
        ),
    )
)

common_truth_article_summary.to_csv(
    RESULTS_DIR
    / "article_common_truth_summary.csv",
    index=False,
)

print("Saved article-facing summaries.")


## 17. Questions This Benchmark Answers

Use the full \(30\)-run results to answer:

1. Does estimator ranking change as the number of coefficients increases?
2. How does RANSAC behave as its minimal sample grows from \(3\) to \(4\) to \(8\)?
3. Do richer models require more measurements before performance stabilizes?
4. Does local B-spline support change how far localized contamination deforms the fit?
5. Does computational cost change enough to alter the practical trade-offs?

Use the **common-truth experiment** for claims specifically about model richness, because all three bases are then fitted to exactly the same underlying quadratic relationship and the same contaminated realizations.

Use the main model-specific benchmark for broader statements about how each curve representation behaves under contamination.

### Important reporting point

The objective is **not** to create a new estimator implementation for Article 2. The basis is changed while the Article 1 robust-estimation machinery is retained.

For the final article, use `RUN_PROFILE = "full"` and base conclusions on distributions across the 30 Monte Carlo realizations rather than individual runs.
